# Notebook PSD — Klasifikasi Audio Berisik dan Non-berisik

Notebook ini membaca dataset fitur yang dibersihkan, meninjau distribusi kelas, menampilkan hasil evaluasi 5-fold StratifiedGroupKFold, dan melihat feature importance. Untuk melatih ulang model dan menulis ulang seluruh artefak, jalankan `python train_model.py` dari root proyek.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import json

ROOT = Path.cwd()
data = pd.read_csv(ROOT / "data" / "features_model.csv")
metadata = json.loads((ROOT / "models" / "model_metadata.json").read_text())

print("Jumlah sampel:", len(data))
print("Jumlah fitur:", len(metadata["feature_names"]))
print("Distribusi kelas:")
display(data["target"].value_counts())

In [ ]:
ax = data["target"].value_counts().reindex(["berisik", "non_berisik"]).plot(kind="bar")
ax.set_title("Distribusi Kelas Setelah Deduplikasi")
ax.set_xlabel("Kelas")
ax.set_ylabel("Jumlah sampel")
plt.tight_layout()
plt.show()

In [ ]:
display(data[metadata["feature_names"]].describe().T)

In [ ]:
evaluation = metadata["evaluation"]
print("Metode:", evaluation["method"])
display(pd.DataFrame([evaluation["metrics"]]).T.rename(columns={0: "Nilai"}))
display(pd.DataFrame(evaluation["classification_report"]).T)

In [ ]:
cm = evaluation["confusion_matrix"]
plt.imshow(cm, interpolation="nearest")
plt.xticks([0, 1], ["Non-berisik", "Berisik"])
plt.yticks([0, 1], ["Non-berisik", "Berisik"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i][j]), ha="center", va="center")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.title("Confusion Matrix — Grouped 5-Fold OOF")
plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.read_csv(ROOT / "reports" / "feature_importance.csv").head(10).sort_values("importance")
plt.barh(importance["feature"], importance["importance"])
plt.xlabel("Importance")
plt.title("10 Fitur Terpenting pada Random Forest")
plt.tight_layout()
plt.show()

## Catatan interpretasi

- Dataset awal memuat 200 file (100 berisik dan 100 non-berisik).
- Setelah menghapus konten identik dalam kelas dan mengecualikan satu konflik label lintas kelas, data modeling berisi 176 sampel seimbang.
- Validasi memakai ID sumber/mahasiswa sebagai group agar sampel dari sumber yang sama tidak masuk ke train dan validasi sekaligus.
- Hasil OOF adalah estimasi awal terhadap label sumber. Label semantik belum diverifikasi manual satu per satu.
- Model aplikasi live menggunakan 16 fitur mean dengan sample rate mono 22.050 Hz.